## PHASE 1

* Load Multi-XScience via Hugging Face, inspect structure, clean text (remove citation markers like @cite_1, extra whitespace/artifacts) 
* Organize into a clean input papers (cluster) → target summary format we'll reuse in every later phase.

In [1]:
import os

for root, dirs, files in os.walk("Multi-XScience"):
    for f in files:
        print(os.path.join(root, f))

Multi-XScience\LICENSE
Multi-XScience\README.md
Multi-XScience\.git\config
Multi-XScience\.git\description
Multi-XScience\.git\HEAD
Multi-XScience\.git\index
Multi-XScience\.git\packed-refs
Multi-XScience\.git\hooks\applypatch-msg.sample
Multi-XScience\.git\hooks\commit-msg.sample
Multi-XScience\.git\hooks\fsmonitor-watchman.sample
Multi-XScience\.git\hooks\post-update.sample
Multi-XScience\.git\hooks\pre-applypatch.sample
Multi-XScience\.git\hooks\pre-commit.sample
Multi-XScience\.git\hooks\pre-merge-commit.sample
Multi-XScience\.git\hooks\pre-push.sample
Multi-XScience\.git\hooks\pre-rebase.sample
Multi-XScience\.git\hooks\pre-receive.sample
Multi-XScience\.git\hooks\prepare-commit-msg.sample
Multi-XScience\.git\hooks\push-to-checkout.sample
Multi-XScience\.git\hooks\sendemail-validate.sample
Multi-XScience\.git\hooks\update.sample
Multi-XScience\.git\info\exclude
Multi-XScience\.git\logs\HEAD
Multi-XScience\.git\logs\refs\heads\master
Multi-XScience\.git\logs\refs\remotes\origin\HEA

In [ ]:
import gzip
import json

def load_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        data = json.load(f) 
    return data

train_data = load_json_gz("Multi-XScience/data/train.json.gz")
val_data = load_json_gz("Multi-XScience/data/val.json.gz")
test_data = load_json_gz("Multi-XScience/data/test.json.gz")

print("Train size:", len(train_data))
print("Val size:", len(val_data))
print("Test size:", len(test_data))

Train size: 30369
Val size: 5066
Test size: 5093


In [6]:
example = train_data[0]

print("TARGET PAPER ABSTRACT:\n", example["abstract"])
print("\nRELATED WORK (gold summary):\n", example["related_work"])
print("\nCITED PAPERS (input cluster):")
for cite_key, cite_data in example["ref_abstract"].items():
    print(f"\n[{cite_key}]:", cite_data["abstract"][:200], "...")

TARGET PAPER ABSTRACT:
 Author(s): Kuperberg, Greg; Thurston, Dylan P. | Abstract: We give a purely topological definition of the perturbative quantum invariants of links and 3-manifolds associated with Chern-Simons field theory. Our definition is as close as possible to one given by Kontsevich. We will also establish some basic properties of these invariants, in particular that they are universally finite type with respect to algebraically split surgery and with respect to Torelli surgery. Torelli surgery is a mutual generalization of blink surgery of Garoufalidis and Levine and clasper surgery of Habiro.

RELATED WORK (gold summary):
 Two other generalizations that can be considered are invariants of graphs in 3-manifolds, and invariants associated to other flat connections @cite_16 . We will analyze these in future work. Among other things, there should be a general relation between flat bundles and links in 3-manifolds on the one hand and finite covers and branched covers on the ot

In [ ]:
import re

def clean_target_abstract(text):
    """Remove the 'Author(s): ... | Abstract: ...' prefix, keep just the abstract."""
    if "Abstract:" in text:
        text = text.split("Abstract:", 1)[1]
    return text.strip()

def clean_related_work(text):
    """Remove @cite_N markers from the gold summary for baseline training."""
    text = re.sub(r"\s*@cite_\d+\s*", " ", text)  
    text = re.sub(r"\s+([.,])", r"\1", text)       
    text = re.sub(r"\s+", " ", text)            
    return text.strip()

def get_cited_abstracts(ref_abstract_dict):
    """Flatten the ref_abstract dict into a simple list of cited paper abstracts."""
    return [v["abstract"].strip() for v in ref_abstract_dict.values() if v.get("abstract")]

def preprocess_example(example):
    return {
        "target_abstract": clean_target_abstract(example["abstract"]),
        "cited_abstracts": get_cited_abstracts(example["ref_abstract"]),
        "gold_summary": clean_related_work(example["related_work"]),
    }

In [10]:
processed = preprocess_example(train_data[0])

print("TARGET ABSTRACT:\n", processed["target_abstract"])
print("\nCITED ABSTRACTS (input cluster):")
for i, abs_text in enumerate(processed["cited_abstracts"]):
    print(f"[{i}]:", abs_text[:150], "...")
print("\nGOLD SUMMARY (cleaned):\n", processed["gold_summary"])

TARGET ABSTRACT:
 We give a purely topological definition of the perturbative quantum invariants of links and 3-manifolds associated with Chern-Simons field theory. Our definition is as close as possible to one given by Kontsevich. We will also establish some basic properties of these invariants, in particular that they are universally finite type with respect to algebraically split surgery and with respect to Torelli surgery. Torelli surgery is a mutual generalization of blink surgery of Garoufalidis and Levine and clasper surgery of Habiro.

CITED ABSTRACTS (input cluster):
[0]: This note is a sequel to our earlier paper of the same title [4] and describes invariants of rational homology 3-spheres associated to acyclic orthogo ...
[1]: Recently, Mullins calculated the Casson-Walker invariant of the 2-fold cyclic branched cover of an oriented link in S^3 in terms of its Jones polynomi ...

GOLD SUMMARY (cleaned):
 Two other generalizations that can be considered are invariants of grap

In [11]:
def process_and_save(data, split_name):
    processed = [preprocess_example(ex) for ex in data]
    with open(f"processed_{split_name}.json", "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2)
    print(f"{split_name}: {len(processed)} examples saved.")

process_and_save(train_data, "train")
process_and_save(val_data, "val")
process_and_save(test_data, "test")

train: 30369 examples saved.
val: 5066 examples saved.
test: 5093 examples saved.
